# Participant Demographics
Loads valid Prolific IDs from the study data and joins with the Prolific demographic export.

In [ ]:
import csv
import math
from collections import Counter

PARTICIPANTS_CSV = "data_participants.csv"
PROLIFIC_DEMO_CSV = "/Users/sheerkarny/Downloads/prolific_demographic_export_69c85399187694d34f96b151.csv"

# --- Load valid Prolific IDs ---
valid_ids = set()
with open(PARTICIPANTS_CSV, newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        pid = row.get('prolific_id', '').strip()
        if pid:
            valid_ids.add(pid)

print(f"Participants in study data: {len(valid_ids)}")

# --- Join with Prolific demographics ---
ages = []
genders = Counter()

with open(PROLIFIC_DEMO_CSV, newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        pid = row.get('Participant id', '').strip()
        if pid not in valid_ids:
            continue
        age = row.get('Age', '').strip()
        sex = row.get('Sex', '').strip()
        if age and age not in ('DATA_EXPIRED', ''):
            try:
                ages.append(int(age))
            except ValueError:
                pass
        if sex and sex not in ('DATA_EXPIRED', '', 'CONSENT_REVOKED'):
            genders[sex] += 1

# --- Stats ---
mean_age = sum(ages) / len(ages)
sd_age = math.sqrt(sum((a - mean_age) ** 2 for a in ages) / len(ages))

print(f"\nAge (n={len(ages)})")
print(f"  Range : {min(ages)}–{max(ages)}")
print(f"  Mean  : {mean_age:.1f}")
print(f"  SD    : {sd_age:.1f}")

print(f"\nGender")
for g, c in sorted(genders.items()):
    print(f"  {g}: {c}")

print(f"\n--- Paper blurb ---")
gender_parts = ', '.join(f'{c} identifying as {g.lower()}' for g, c in sorted(genders.items()))
print(
    f"The ages of the participants ranged from {min(ages)} to {max(ages)} "
    f"(M = {mean_age:.1f}, SD = {sd_age:.1f}), with {gender_parts}."
)